# PCA of lab IMU embeddings

This notebook runs separate PCA analyses on generic and stroke-adapted LIMU-BERT-X and HART embeddings. Each row is one participant, visit, and NW walking trial. Window embeddings were mean-pooled within each trial before PCA, giving 77 trial rows from 17 participants. Fall-risk labels are shown only as plot annotations and are not PCA inputs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

search_paths = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in search_paths if (p / 'Data/analysis/master_analysis_table_pca.csv').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not find Data/analysis/master_analysis_table_pca.csv from the current directory.')

df = pd.read_csv(repo_root / 'Data/analysis/master_analysis_table_pca.csv')
results_dir = repo_root / 'Data/analysis/pca_results'
results_dir.mkdir(parents=True, exist_ok=True)
print(f'Loaded {len(df)} trial rows from {df.participant_id.nunique()} participants.')

In [ ]:
feature_blocks = {
    'LIMU-BERT-X generic': [c for c in df.columns if c.startswith('limu_generic_embedding_')],
    'LIMU-BERT-X stroke-adapted': [c for c in df.columns if c.startswith('limu_stroke_adapted_embedding_')],
    'HART generic': [c for c in df.columns if c.startswith('hart_generic_embedding_')],
    'HART stroke-adapted': [c for c in df.columns if c.startswith('hart_stroke_adapted_embedding_')],
}
for name, columns in feature_blocks.items():
    print(f'{name}: {len(columns)} embedding dimensions')

In [ ]:
def fit_embedding_pca(name, columns):
    values = df[columns].apply(pd.to_numeric, errors='coerce')
    keep = values.notna().all(axis=1)
    values = values.loc[keep]
    scaler = StandardScaler()
    scaled = scaler.fit_transform(values)
    pca = PCA(svd_solver='full')
    scores = pca.fit_transform(scaled)
    score_df = pd.DataFrame(scores[:, :4], columns=['PC1', 'PC2', 'PC3', 'PC4'], index=values.index)
    score_df['participant_id'] = df.loc[keep, 'participant_id'].to_numpy()
    score_df['fall_risk_label'] = df.loc[keep, 'fall_risk_label'].to_numpy()
    score_df['walk_trial_name'] = df.loc[keep, 'walk_trial_name'].to_numpy()
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    summary = {
        'embedding_block': name,
        'trial_rows': len(values),
        'participants': df.loc[keep, 'participant_id'].nunique(),
        'embedding_dimensions': len(columns),
        'PC1_variance_pct': 100 * pca.explained_variance_ratio_[0],
        'PC2_variance_pct': 100 * pca.explained_variance_ratio_[1],
        'PC1_PC2_cumulative_pct': 100 * cumulative[1],
        'components_for_80pct': int(np.searchsorted(cumulative, .80) + 1),
        'components_for_90pct': int(np.searchsorted(cumulative, .90) + 1),
    }
    return pca, score_df, summary

pca_models = {}
score_tables = {}
summary_rows = []
for name, columns in feature_blocks.items():
    pca_models[name], score_tables[name], summary = fit_embedding_pca(name, columns)
    summary_rows.append(summary)

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(results_dir / 'explained_variance_summary.csv', index=False)
display(summary_table.round(2))

In [ ]:
def show_variance_curve(name):
    ratios = pca_models[name].explained_variance_ratio_
    cumulative = np.cumsum(ratios)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(1, len(cumulative) + 1), 100 * cumulative, marker='.', linewidth=1)
    ax.axhline(80, color='gray', linestyle='--', linewidth=1, label='80%')
    ax.axhline(90, color='gray', linestyle=':', linewidth=1, label='90%')
    ax.set(xlabel='Number of principal components', ylabel='Cumulative explained variance (%)',
           title=f'{name}: cumulative explained variance', ylim=(0, 101))
    ax.grid(alpha=.2)
    ax.legend()
    fig.tight_layout()
    slug=name.lower().replace(' ', '_').replace('-', '_')
    fig.savefig(results_dir / f'{slug}_variance.png', dpi=160)
    plt.show()

def show_score_plot(name):
    scores=score_tables[name]
    labels=scores['fall_risk_label'].fillna('Unclassified')
    fig, ax=plt.subplots(figsize=(7, 5))
    for label in sorted(labels.unique()):
        mask=labels==label
        ax.scatter(scores.loc[mask,'PC1'],scores.loc[mask,'PC2'],label=label,alpha=.8,s=38)
    pca=pca_models[name]
    ax.set(xlabel=f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)',
           ylabel=f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)',
           title=f'{name}: trial-level scores')
    ax.legend(title='Fall-risk label',fontsize=8)
    ax.grid(alpha=.2)
    fig.tight_layout()
    slug=name.lower().replace(' ', '_').replace('-', '_')
    fig.savefig(results_dir / f'{slug}_pc1_pc2.png',dpi=160)
    plt.show()

## LIMU-BERT-X generic

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('LIMU-BERT-X generic')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('LIMU-BERT-X generic')

## LIMU-BERT-X stroke-adapted

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('LIMU-BERT-X stroke-adapted')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('LIMU-BERT-X stroke-adapted')

## HART generic

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('HART generic')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('HART generic')

## HART stroke-adapted

Cumulative variance explained, shown separately for this embedding block.

In [ ]:
show_variance_curve('HART stroke-adapted')

PC1 and PC2 trial scores. Color indicates the participant-level fall-risk label; it was not used to fit PCA.

In [ ]:
show_score_plot('HART stroke-adapted')

## Participant-level correlations with gait measures

For each embedding block, the first four PC scores and gait measures are averaged across a participant's available NW trials. Spearman correlations are then calculated across participants (n=17), so repeated trial rows are not treated as independent observations. Step count is excluded because it is constant in the extracted data. Benjamini–Hochberg adjusted q-values account for the 48 exploratory PC-by-measure comparisons.

In [ ]:
from scipy.stats import spearmanr

pc_names = ['PC1', 'PC2', 'PC3', 'PC4']
gait_targets = {
    'Cadence (steps/min)': 'sacrum_walk_cadence_steps_min',
    'Mean step time (s)': 'sacrum_walk_mean_step_time_s',
    'Extracted walking duration (s)': 'sacrum_walk_duration_s',
}
# Average trial-level gait measurements to one value per participant.
participant_gait = df.groupby('participant_id')[list(gait_targets.values())].mean()
rows = []
for block_name, scores in score_tables.items():
    participant_pc = scores.groupby('participant_id')[pc_names].mean()
    aligned = participant_pc.join(participant_gait, how='inner')
    for pc in pc_names:
        for measure_label, measure_col in gait_targets.items():
            rho, p_value = spearmanr(aligned[pc], aligned[measure_col])
            rows.append({
                'embedding_block': block_name,
                'PC': pc,
                'gait_measure': measure_label,
                'n_participants': len(aligned),
                'spearman_rho': rho,
                'p_value': p_value,
            })
correlations = pd.DataFrame(rows)
# Benjamini-Hochberg adjustment across all 48 exploratory correlations.
order = np.argsort(correlations['p_value'].to_numpy())
p_sorted = correlations['p_value'].to_numpy()[order]
m = len(p_sorted)
q_sorted = np.minimum.accumulate((p_sorted * m / np.arange(1, m + 1))[::-1])[::-1]
q_values = np.empty(m)
q_values[order] = np.clip(q_sorted, 0, 1)
correlations['fdr_q_value'] = q_values
correlations.to_csv(results_dir / 'pca_pc_gait_correlations_subject_level.csv', index=False)
print(f'Participant-level correlation sample size: {correlations.n_participants.min()}–{correlations.n_participants.max()} participants.')
print('Step count excluded because it is constant at 10 in the extracted gait data.')

### LIMU-BERT-X generic

In [ ]:
display(correlations.loc[correlations.embedding_block == 'LIMU-BERT-X generic', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### LIMU-BERT-X stroke-adapted

In [ ]:
display(correlations.loc[correlations.embedding_block == 'LIMU-BERT-X stroke-adapted', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### HART generic

In [ ]:
display(correlations.loc[correlations.embedding_block == 'HART generic', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

### HART stroke-adapted

In [ ]:
display(correlations.loc[correlations.embedding_block == 'HART stroke-adapted', ['PC', 'gait_measure', 'n_participants', 'spearman_rho', 'p_value', 'fdr_q_value']].round(3).reset_index(drop=True))

## Interpretation notes

- These PCA and correlation analyses are exploratory. Participants contribute repeated trial records to the PCA; the correlations use one participant-level average per person.
- Fall-risk labels only annotate the PCA score plots and are not used to fit PCA or these correlations.
- The gait measures are strongly related, and step count is constant in this extracted dataset. Interpret correlations as associations with these recorded measures, not independent confirmations of gait constructs.
- Correlation does not establish clinical validity or predictive performance. The small sample (17 participants) limits precision.